In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

instagram_df = pd.read_csv(PROCESSED / "instagram_processed.csv")
instafake_df = pd.read_csv(PROCESSED / "instafake_processed.csv")
cresci_df = pd.read_csv(PROCESSED / "cresci2017_processed.csv")

print("Instagram:", instagram_df.shape)
print("InstaFake:", instafake_df.shape)
print("Cresci:", cresci_df.shape)

Instagram: (696, 8)
InstaFake: (1194, 8)
Cresci: (4465, 9)


In [20]:
def add_follower_following_ratio(df):
    df = df.copy()

    df["follower_following_ratio"] = (
        df["followers"] /
        df["following"].replace(0, np.nan)
    )

    # Replace invalid/infinite values
    df["follower_following_ratio"] = (
        df["follower_following_ratio"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return df


instagram_df = add_follower_following_ratio(instagram_df)
instafake_df = add_follower_following_ratio(instafake_df)
cresci_df = add_follower_following_ratio(cresci_df)

In [21]:
def add_profile_completeness(df):
    df = df.copy()

    profile_fields = [
        "followers",
        "following",
        "posts",
        "bio_length",
        "has_profile_picture"
    ]

    available = df[profile_fields].notna().sum(axis=1)

    df["profile_completeness"] = (
        available / len(profile_fields)
    )

    return df


instagram_df = add_profile_completeness(instagram_df)
instafake_df = add_profile_completeness(instafake_df)
cresci_df = add_profile_completeness(cresci_df)

In [22]:
for name, df in {
    "Instagram": instagram_df,
    "InstaFake": instafake_df,
    "Cresci": cresci_df
}.items():

    print(f"\n{name}")
    print("-" * 40)
    print(
        df[
            [
                "follower_following_ratio",
                "profile_completeness"
            ]
        ].describe()
    )


Instagram
----------------------------------------
       follower_following_ratio  profile_completeness
count              6.960000e+02                 696.0
mean               3.006333e+03                   1.0
std                5.974643e+04                   0.0
min                0.000000e+00                   1.0
25%                3.419083e-01                   1.0
50%                7.854740e-01                   1.0
75%                1.621307e+00                   1.0
max                1.549715e+06                   1.0

InstaFake
----------------------------------------
       follower_following_ratio  profile_completeness
count               1194.000000                1194.0
mean                   0.810519                   1.0
std                    0.805207                   0.0
min                    0.000000                   1.0
25%                    0.480396                   1.0
50%                    0.838178                   1.0
75%                    1.003664 

In [23]:
for name, df in {
    "Instagram": instagram_df,
    "InstaFake": instafake_df,
    "Cresci": cresci_df
}.items():

    print(
        name,
        "NaN:",
        df.isna().sum().sum(),
        "| Inf:",
        np.isinf(df.select_dtypes(include=np.number)).sum().sum()
    )

Instagram NaN: 0 | Inf: 0
InstaFake NaN: 0 | Inf: 0
Cresci NaN: 8852 | Inf: 0


In [24]:
instagram_df.to_csv(
    PROCESSED / "instagram_features.csv",
    index=False
)

instafake_df.to_csv(
    PROCESSED / "instafake_features.csv",
    index=False
)

cresci_df.to_csv(
    PROCESSED / "cresci2017_features.csv",
    index=False
)

print("✅ Feature engineering complete.")

✅ Feature engineering complete.
